# **Step 2: Add Demographic Data to PPG Signals**

**Input:** `all-ppg-data.csv` (from Step 1) + MIMIC-III clinical tables (PATIENTS, ADMISSIONS, DIAGNOSES_ICD)

This notebook enriches raw PPG waveforms with demographic information (age, gender) and extracts diagnosis events separately for time-aware disease labeling in Step 3.

## Outputs
1. **master-demographic-ppg-data-no-icd.csv** → PPG + age + gender (ready for Step 3 labeling)
2. **diagnosis-events-for-time-aware-labeling.csv** → Diagnosis metadata for temporal label alignment

---
## **Import Libraries**

In [ ]:
import os
import numpy as np
import pandas as pd

## **Load Datasets**

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
CLINICAL_DIR = PROJECT_ROOT / "data" / "mimic_clinical"

# Inputs generated by Step 1 and credentialed MIMIC-III clinical tables.
ppg_df = pd.read_csv(INTERMEDIATE_DIR / "all-ppg-data.csv")
patients_df = pd.read_csv(CLINICAL_DIR / "PATIENTS.csv")
admissions_df = pd.read_csv(CLINICAL_DIR / "ADMISSIONS.csv")
diagnoses_df = pd.read_csv(CLINICAL_DIR / "DIAGNOSES_ICD.csv")

print(f"PPG rows: {len(ppg_df)}")
print(f"Patients rows: {len(patients_df)}")
print(f"Admissions rows: {len(admissions_df)}")
print(f"Diagnoses rows: {len(diagnoses_df)}")

## **Validate Step 1 Output**

In [ ]:
# Validate PPG dataset from Step 1
required_ppg_columns = {
    "patient_id", "group", "record_id", "fs",
    "ppg_min_3", "ppg_min_4", "ppg_min_5", 
    "ppg_min_6", "ppg_min_7", "ppg_min_8"
}

missing = required_ppg_columns - set(ppg_df.columns)
if missing:
    raise ValueError(f"❌ Step 1 output missing columns: {missing}")

if len(ppg_df) == 0:
    raise ValueError("❌ Step 1 output is empty (0 rows)")

print(f"✅ Step 1 output validation passed")
print(f"   Rows: {len(ppg_df)}")
print(f"   Columns: {len(ppg_df.columns)}")
print(f"   Required columns verified: {len(required_ppg_columns)}/10")

## **Derive SUBJECT_ID from patient_id**

In [ ]:
# patient_id format: p010023 -> SUBJECT_ID = 10023
ppg_df["SUBJECT_ID"] = ppg_df["patient_id"].str[1:].astype(int)
ppg_df[["patient_id", "SUBJECT_ID"]].head()

## **Attach gender**

In [ ]:
ppg_df = ppg_df.merge(
    patients_df[["SUBJECT_ID", "GENDER"]],
    on="SUBJECT_ID",
    how="left"
)

print(ppg_df[["SUBJECT_ID", "GENDER"]].head())

## **Compute age with MIMIC-III de-identification handling**

In [ ]:
patients_df["DOB"] = pd.to_datetime(patients_df["DOB"], errors="coerce")
admissions_df["ADMITTIME"] = pd.to_datetime(admissions_df["ADMITTIME"], errors="coerce")

first_admit = (
    admissions_df
    .sort_values("ADMITTIME")
    .groupby("SUBJECT_ID", as_index=False)
    .first()[["SUBJECT_ID", "ADMITTIME"]]
)

age_df = first_admit.merge(
    patients_df[["SUBJECT_ID", "DOB"]],
    on="SUBJECT_ID",
    how="left"
)

# Standard MIMIC-style year difference.
age_df["AGE"] = age_df["ADMITTIME"].dt.year - age_df["DOB"].dt.year

# Clean impossible values and map de-identified elderly ages.
age_df.loc[age_df["AGE"] < 0, "AGE"] = np.nan
age_df.loc[age_df["AGE"] > 300, "AGE"] = 90

# Step 1 output never has AGE, so merge will create exactly one AGE column
ppg_df = ppg_df.merge(
    age_df[["SUBJECT_ID", "AGE"]],
    on="SUBJECT_ID",
    how="left"
)

print(ppg_df[["SUBJECT_ID", "AGE"]].head())

## **Prepare diagnosis events for later time-aware labeling**

In [ ]:
diagnosis_events = diagnoses_df[["SUBJECT_ID", "HADM_ID", "SEQ_NUM", "ICD9_CODE"]].copy()
diagnosis_events = diagnosis_events.dropna(subset=["SUBJECT_ID", "ICD9_CODE"])
diagnosis_events["ICD9_CODE"] = diagnosis_events["ICD9_CODE"].astype(str).str.strip()
diagnosis_events = diagnosis_events.sort_values(["SUBJECT_ID", "HADM_ID", "SEQ_NUM"], na_position="last").reset_index(drop=True)

print(f"Diagnosis events prepared: {len(diagnosis_events)} rows")
diagnosis_events.head()

## **Final column order for demographic PPG dataset**

In [ ]:
final_cols = [
    "patient_id",
    "SUBJECT_ID",
    "group",
    "record_id",
    "fs",
    "AGE",
    "GENDER",
    "ppg_min_3",
    "ppg_min_4",
    "ppg_min_5",
    "ppg_min_6",
    "ppg_min_7",
    "ppg_min_8"
]

missing = [c for c in final_cols if c not in ppg_df.columns]
print("Missing columns:", missing)

ppg_master = ppg_df[final_cols].copy()
ppg_master.head()

## **Save outputs**

## **Verify outputs**

Display the tail of both exported datasets to confirm data integrity.

In [ ]:
output_ppg = INTERMEDIATE_DIR / "master-demographic-ppg-data-no-icd.csv"
output_diag = INTERMEDIATE_DIR / "diagnosis-events-for-time-aware-labeling.csv"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

ppg_master.to_csv(output_ppg, index=False)
diagnosis_events.to_csv(output_diag, index=False)

print(f"Saved demographic PPG dataset with {len(ppg_master)} rows")
print(f"File saved at: {output_ppg}")
print(f"Saved diagnosis-events dataset with {len(diagnosis_events)} rows")
print(f"File saved at: {output_diag}")

In [ ]:
ppg_master.tail()

In [ ]:
diagnosis_events.tail()